# Rhylthyme Timeline in a Jupyter Notebook

This notebook demonstrates how to load a Rhylthyme program and render its interactive D3 timeline visualization inline.

In [ ]:
import json
from IPython.display import HTML, display
from rhylthyme_web.web.web_visualizer import (
    extract_step_dependencies,
    generate_dag_html,
)
from rhylthyme_web.rhylthyme import load_program_file

## Load a program

Load one of the example programs and inspect its structure.

In [ ]:
program = load_program_file("programs/bakery_program_example.json")
print(f"Program: {program['name']}")
print(f"Tracks: {len(program['tracks'])}")
for track in program["tracks"]:
    print(f"  - {track['name']}: {len(track['steps'])} steps")

## Render the timeline

Generate the visualization HTML and display it in an iframe.

In [ ]:
def show_timeline(program, height=600):
    """Render a Rhylthyme program as an interactive timeline in the notebook."""
    nodes, edges = extract_step_dependencies(program)
    resource_constraints = program.get("resourceConstraints", [])
    html = generate_dag_html(nodes, edges, program, None, resource_constraints)
    # Embed in an iframe via srcdoc
    import html as html_mod
    escaped = html_mod.escape(html)
    iframe = f'<iframe srcdoc="{escaped}" style="width:100%;height:{height}px;border:1px solid #ccc;border-radius:8px;" allowfullscreen></iframe>'
    display(HTML(iframe))

In [ ]:
show_timeline(program)

## Build a program from scratch

You can also define a program as a Python dictionary and visualize it directly.

In [ ]:
my_program = {
    "programId": "morning-coffee",
    "name": "Morning Coffee",
    "version": "0.1.0",
    "tracks": [
        {
            "trackId": "brew",
            "name": "Brew Coffee",
            "steps": [
                {
                    "stepId": "grind",
                    "name": "Grind Beans",
                    "task": "grinder",
                    "trigger": {"type": "programStart"},
                    "duration": {"type": "fixed", "seconds": 30},
                },
                {
                    "stepId": "pour-over",
                    "name": "Pour Over",
                    "task": "kettle",
                    "trigger": {"on": "grind"},
                    "duration": {"type": "fixed", "seconds": 240},
                },
            ],
        },
        {
            "trackId": "water",
            "name": "Heat Water",
            "steps": [
                {
                    "stepId": "boil",
                    "name": "Boil Water",
                    "task": "kettle",
                    "trigger": {"type": "programStart"},
                    "duration": {"type": "variable", "minSeconds": 120, "maxSeconds": 180, "defaultSeconds": 150},
                },
            ],
        },
    ],
    "resourceConstraints": [
        {"task": "kettle", "maxConcurrent": 1},
        {"task": "grinder", "maxConcurrent": 1},
    ],
}

show_timeline(my_program)